In [1]:
# Cell 1: Environment and Visualization Setup
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set cohesive aesthetic styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Output directory for portfolio screenshots
SCREENSHOTS_DIR = Path("../dashboard/screenshots")
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualizations will be saved to:", SCREENSHOTS_DIR.resolve())

Visualizations will be saved to: C:\Users\yashn\Olist E-Commerce Analysis\dashboard\screenshots


In [2]:
# Cell 2: Load All 10 SQL Processed CSV Datasets
DATA_DIR = Path("../data/processed")

csv_registry = {
    "monthly_revenue": ("monthly_revenue.csv", ["order_month"]),
    "category_trend": ("category_trend.csv", []),
    "top_products": ("top_products_per_category.csv", []),
    "cohort_retention": ("cohort_retention.csv", ["cohort_month", "order_month"]),
    "delivery_reviews": ("delivery_vs_reviews.csv", []),
    "late_by_state": ("late_by_state.csv", []),
    "freight_burden": ("freight_burden.csv", []),
    "repeat_rate": ("repeat_rate.csv", []),
    "pareto": ("pareto.csv", []),
    "seller_late": ("seller_late_rate.csv", []),
}

dfs = {}
for key, (filename, date_cols) in csv_registry.items():
    file_path = DATA_DIR / filename
    if file_path.exists():
        dfs[key] = pd.read_csv(file_path, parse_dates=date_cols if date_cols else False)
        print(f"[LOADED] {filename.ljust(30)} ({len(dfs[key]):,} rows)")
    else:
        print(f"[WAITING] {filename.ljust(30)} (Not found in data/processed/)")
        dfs[key] = pd.DataFrame()

[LOADED] monthly_revenue.csv            (20 rows)
[LOADED] category_trend.csv             (74 rows)
[LOADED] top_products_per_category.csv  (220 rows)
[LOADED] cohort_retention.csv           (219 rows)
[LOADED] delivery_vs_reviews.csv        (2 rows)
[LOADED] late_by_state.csv              (27 rows)
[LOADED] freight_burden.csv             (24 rows)
[LOADED] repeat_rate.csv                (1 rows)
[LOADED] pareto.csv                     (93,104 rows)
[LOADED] seller_late_rate.csv           (460 rows)


In [3]:
# Cell 3: DATA QUALITY LOG (README Source of Truth)
print("=" * 70)
print("DATA QUALITY & CORE PIPELINE LOG")
print("=" * 70)

if not dfs["monthly_revenue"].empty and not dfs["repeat_rate"].empty:
    total_delivered_orders = dfs["monthly_revenue"]["total_orders"].sum()
    min_date = dfs["monthly_revenue"]["order_month"].min().strftime("%Y-%m-%d")
    max_date = dfs["monthly_revenue"]["order_month"].max().strftime("%Y-%m-%d")
    rep_df = dfs["repeat_rate"].iloc[0]
    
    print(f"Delivered Order Count (Clean Window): {total_delivered_orders:,}")
    print(f"Date Range Analyzed:                  {min_date} to {max_date}")
    print(f"Unique Customers (Human Individuals): {rep_df['total_unique_customers']:,}")
    print(f"One-Time Buyers:                      {rep_df['one_time_buyers']:,}")
    print(f"Repeat Buyers (>1 order):             {rep_df['repeat_buyers']:,}")
    print(f"Repeat Purchase Rate:                 {rep_df['repeat_purchase_rate_pct']:.2f}%")
    print(f"Null Delivery Date Filtered Orders:   Approx. 8 delivered orders")
else:
    print("STATUS: Waiting for processed CSVs to be exported from PostgreSQL.")
    print("Expected metrics to verify:")
    print("  - Delivered Order Count (2017-01 to 2018-08): ~96,000+")
    print("  - Unique customer_id (~99,441) vs customer_unique_id (~96,096)")
    print("  - Repeat purchase rate: ~3.0%")
    print("  - Null delivery dates: ~8 delivered records")
print("=" * 70)

DATA QUALITY & CORE PIPELINE LOG
Delivered Order Count (Clean Window): 96,211
Date Range Analyzed:                  2017-01-01 to 2018-08-01
Unique Customers (Human Individuals): 93,104.0
One-Time Buyers:                      90,315.0
Repeat Buyers (>1 order):             2,789.0
Repeat Purchase Rate:                 3.00%
Null Delivery Date Filtered Orders:   Approx. 8 delivered orders


In [4]:
# Cell 4: Monthly Revenue Line with Secondary Axis for Orders
df_rev = dfs.get("monthly_revenue")
if not df_rev.empty:
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    color_rev = "#1f77b4"
    color_ord = "#ff7f0e"
    
    x_dates = df_rev["order_month"].dt.strftime("%Y-%m")
    
    ax1.set_xlabel("Order Month", fontsize=12)
    ax1.set_ylabel("Gross Revenue (BRL)", color=color_rev, fontsize=12)
    line1 = ax1.plot(x_dates, df_rev["total_revenue"], color=color_rev, marker="o", linewidth=2.5, label="Monthly Revenue")
    ax1.tick_params(axis="y", labelcolor=color_rev)
    ax1.tick_params(axis="x", rotation=45)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    ax2 = ax1.twinx()
    ax2.set_ylabel("Order Count", color=color_ord, fontsize=12)
    line2 = ax2.plot(x_dates, df_rev["total_orders"], color=color_ord, marker="s", linestyle="--", linewidth=2, label="Order Count")
    ax2.tick_params(axis="y", labelcolor=color_ord)
    ax2.grid(False)
    
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left", frameon=True)
    
    plt.title("Olist Monthly Revenue & Order Growth (Jan 2017 - Aug 2018)", pad=15)
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "01_monthly_revenue.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for monthly_revenue.csv not found.")

In [5]:
# Cell 5: Top 10 Categories Horizontal Bar Chart
df_cat = dfs.get("category_trend")
if not df_cat.empty:
    top10 = df_cat.head(10).sort_values("revenue_h1_2018", ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(top10["category_name"], top10["revenue_h1_2018"] / 1e3, color="#2ca02c", alpha=0.85)
    
    ax.set_xlabel("H1 2018 Revenue (in Thousands BRL)", fontsize=12)
    ax.set_ylabel("Product Category", fontsize=12)
    ax.set_title("Top 10 Product Categories by H1 2018 Revenue", pad=15)
    
    for bar in bars:
        width = bar.get_width()
        ax.annotate(f"R$ {width:,.1f}k",
                    xy=(width, bar.get_y() + bar.get_height() / 2),
                    xytext=(5, 0), textcoords="offset points",
                    ha="left", va="center", fontsize=10)
        
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "02_top_categories.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for category_trend.csv not found.")

In [6]:
# Cell 6: Review Score by Delivery Performance (Late vs On-Time)
df_deliv = dfs.get("delivery_reviews")
if not df_deliv.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#d62728" if x == "Late" else "#2ca02c" for x in df_deliv["delivery_performance"]]
    bars = ax.bar(df_deliv["delivery_performance"], df_deliv["avg_review_score"], color=colors, width=0.5)
    
    ax.set_ylabel("Average Customer Review Score (1-5 Scale)", fontsize=12)
    ax.set_ylim(0, 5.0)
    ax.set_title("Impact of Delivery Timeliness on Customer Review Scores", pad=15)
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f} ★",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha="center", va="bottom", fontsize=13, fontweight="bold")
        
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "03_delivery_vs_reviews.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for delivery_vs_reviews.csv not found.")

In [7]:
# Cell 7: Cohort Retention Heatmap
df_cohort = dfs.get("cohort_retention")
if not df_cohort.empty:
    # Pivot table to cohort matrix
    df_cohort["cohort_month_str"] = df_cohort["cohort_month"].dt.strftime("%Y-%m")
    pivot_cust = df_cohort.pivot(index="cohort_month_str", columns="months_since_first", values="customers")
    
    # Calculate cohort initial sizes and retention percentages
    cohort_sizes = pivot_cust.iloc[:, 0]
    retention_matrix = pivot_cust.divide(cohort_sizes, axis=0) * 100.0
    
    # Calculate average Month 1 retention rate across eligible cohorts
    m1_pct = retention_matrix[1].mean() if 1 in retention_matrix.columns else 0.0
    
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(
        retention_matrix.iloc[:, :12], 
        annot=True, 
        fmt=".1f", 
        cmap="Blues", 
        cbar_kws={"label": "Retention Rate (%)"}, 
        linewidths=0.5,
        ax=ax
    )
    ax.set_title(f"Monthly Customer Cohort Retention (Average Month 1 Retention: {m1_pct:.2f}%)", pad=15)
    ax.set_xlabel("Months Since First Purchase", fontsize=12)
    ax.set_ylabel("Cohort Month", fontsize=12)
    
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "04_cohort_retention.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for cohort_retention.csv not found.")

In [8]:
# Cell 8: Pareto Cumulative Curve
df_pareto = dfs.get("pareto")
if not df_pareto.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(df_pareto["customer_percentile"], df_pareto["cumulative_revenue_share_pct"], 
            color="#1f77b4", linewidth=2.5, label="Actual Cumulative Revenue Share")
    ax.plot([0, 100], [0, 100], linestyle="--", color="grey", alpha=0.7, label="Equal Revenue Line (45°)")
    
    # Identify where cumulative revenue crosses 80%
    cross_80 = df_pareto[df_pareto["cumulative_revenue_share_pct"] >= 80.0].iloc[0]
    cross_cust_pct = cross_80["customer_percentile"]
    
    ax.axhline(80, color="#d62728", linestyle=":", linewidth=1.5)
    ax.axvline(cross_cust_pct, color="#d62728", linestyle=":", linewidth=1.5)
    
    ax.scatter([cross_cust_pct], [80], color="#d62728", s=80, zorder=5)
    ax.annotate(f"80% Revenue at {cross_cust_pct:.1f}% Customers\n(Traditional Pareto expectation: 20%)",
                xy=(cross_cust_pct, 80),
                xytext=(cross_cust_pct - 35, 86),
                arrowprops=dict(facecolor="black", shrink=0.05, width=1, headwidth=6),
                fontsize=11, fontweight="bold", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#d62728"))
    
    ax.set_xlabel("Customer Percentile (%)", fontsize=12)
    ax.set_ylabel("Cumulative Revenue Share (%)", fontsize=12)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 105)
    ax.set_title("Customer Revenue Pareto Curve (Testing the 80/20 Rule)", pad=15)
    ax.legend(loc="lower right")
    
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "05_pareto.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for pareto.csv not found.")

In [9]:
# Cell 9: Review Score Distribution (Showing Bimodality)
df_deliv = dfs.get("delivery_reviews")
if not df_deliv.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    perf_types = df_deliv["delivery_performance"].tolist()
    x = np.arange(len(perf_types))
    width = 0.35
    
    rects1 = ax.bar(x - width/2, df_deliv["pct_5_star"], width, label="5-Star (Delighted)", color="#2ca02c")
    rects2 = ax.bar(x + width/2, df_deliv["pct_1_star"], width, label="1-Star (Severely Dissatisfied)", color="#d62728")
    
    ax.set_ylabel("Percentage of Reviews (%)", fontsize=12)
    ax.set_title("Polarized Review Score Distribution: 5-Star vs 1-Star Share", pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(perf_types, fontsize=12)
    ax.legend()
    
    for rect in rects1 + rects2:
        h = rect.get_height()
        ax.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom", fontsize=10)
                    
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "07_review_distribution.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for delivery_vs_reviews.csv not found.")

In [10]:
# Cell 10: Top 10 States by Order Volume and Revenue
df_state = dfs.get("late_by_state")
if not df_state.empty:
    top10_states = df_state.sort_values("total_orders", ascending=False).head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(top10_states["customer_state"], top10_states["total_orders"], color="#1f77b4", alpha=0.85)
    
    ax.set_xlabel("Customer State (Federative Unit)", fontsize=12)
    ax.set_ylabel("Total Orders Delivered", fontsize=12)
    ax.set_title("Top 10 Brazilian States by Delivered Order Volume", pad=15)
    
    for bar in bars:
        yval = bar.get_height()
        ax.annotate(f"{yval:,.0f}", xy=(bar.get_x() + bar.get_width() / 2, yval),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=10)
                    
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "06_revenue_by_state.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for late_by_state.csv not found.")

In [11]:
# Cell 11: Actual vs Estimated Delivery Days by State
df_state = dfs.get("late_by_state")
if not df_state.empty:
    top_late_states = df_state.sort_values("late_rate_pct", ascending=False).head(10)
    
    x = np.arange(len(top_late_states))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    rects1 = ax.bar(x - width/2, top_late_states["avg_delivery_days"], width, label="Actual Delivery Days", color="#d62728")
    rects2 = ax.bar(x + width/2, top_late_states["avg_estimated_days"], width, label="Estimated Delivery Days", color="#1f77b4")
    
    ax.set_xlabel("Customer State (Top 10 by Late Rate)", fontsize=12)
    ax.set_ylabel("Average Duration (Days)", fontsize=12)
    ax.set_title("Actual vs Estimated Transit Time for States with Highest Delay Rates", pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(top_late_states["customer_state"], fontsize=11)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(SCREENSHOTS_DIR / "08_delivery_days_by_state.png", dpi=150)
    plt.show()
else:
    print("[WAITING] Data for late_by_state.csv not found.")

In [12]:
# Cell 12: EXECUTIVE SUMMARY & KEY FINDINGS
print("=" * 80)
print("                      EXECUTIVE PORTFOLIO FINDINGS")
print("=" * 80)
print("1. REVENUE GROWTH & SEASONALITY:")
print("   Gross marketplace sales expanded rapidly throughout 2017 and peaked in November 2017")
print("   at R$ 987,765.37 across 7,289 orders (+52.37% MoM), before stabilizing around R$ 902k/mo in 2018.")
print("")
print("2. CUSTOMER RETENTION DEFICIT:")
print("   The marketplace operates almost entirely on single-transaction customer acquisitions;")
print("   repeat purchase rate is 3.00% (2,789 repeat buyers) with Month 1 cohort retention")
print("   averaging just 0.48% across all monthly cohorts.")
print("")
print("3. SEVERE LOGISTICS REPUTATIONAL RISK:")
print("   Late delivery inflicts massive reputational damage, dropping average customer review scores")
print("   from 4.29 to 2.57 stars (-1.72 drop), with 1-star reviews surging from 6.58% to 46.15%.")
print("")
print("4. GEOGRAPHIC LOGISTICS FRICTION:")
print("   Logistics delays are concentrated in remote Northern and Northeastern states, where transit")
print("   times reach 24.52 to 29.91 days with late rates in Alagoas (AL) reaching 23.99%.")
print("")
print("5. DISPROPORTIONATE FREIGHT BURDEN:")
print("   Shipping fees represent 51.53% to 59.58% of product value for remote consumers (e.g. RO, MA, PB),")
print("   compared to 26.51% in Sao Paulo, creating steep regional friction.")
print("")
print("6. THE PARETO MISCONCEPTION:")
print("   The customer revenue distribution does not exhibit an 80/20 rule: the top 20% of customers")
print("   generate only 56.61% of revenue, and it takes 45.00% of customers to reach 80% of revenue.")
print("   Growth depends on customer acquisition volume rather than an entrenched VIP luxury cohort.")
print("=" * 80)


                      EXECUTIVE PORTFOLIO FINDINGS
1. REVENUE GROWTH & SEASONALITY:
   Gross marketplace sales expanded rapidly throughout 2017 and peaked in November 2017
   at R$ 987,765.37 across 7,289 orders (+52.37% MoM), before stabilizing around R$ 902k/mo in 2018.

2. CUSTOMER RETENTION DEFICIT:
   The marketplace operates almost entirely on single-transaction customer acquisitions;
   repeat purchase rate is 3.00% (2,789 repeat buyers) with Month 1 cohort retention
   averaging just 0.48% across all monthly cohorts.

3. SEVERE LOGISTICS REPUTATIONAL RISK:
   Late delivery inflicts massive reputational damage, dropping average customer review scores
   from 4.29 to 2.57 stars (-1.72 drop), with 1-star reviews surging from 6.58% to 46.15%.

4. GEOGRAPHIC LOGISTICS FRICTION:
   Logistics delays are concentrated in remote Northern and Northeastern states, where transit
   times reach 24.52 to 29.91 days with late rates in Alagoas (AL) reaching 23.99%.

5. DISPROPORTIONATE FREIGHT 

In [13]:
# Cell 13: Matched-Pair Causal Analysis Summary
matched_file = Path("../data/processed/late_delivery_matched.csv")
if matched_file.exists():
    df_matched = pd.read_csv(matched_file)
    n_pairs = len(df_matched)
    n_unique_late = df_matched["late_order_id"].nunique()
    avg_late = df_matched["late_score"].mean()
    avg_ontime = df_matched["ontime_score"].mean()
    delta = avg_late - avg_ontime
    
    print("=" * 80)
    print("             CAUSAL INFERENCE: MATCHED-PAIR ANALYSIS (LATE VS ON-TIME)")
    print("=" * 80)
    print(f"Total Matched Pairs (same state, category/product, +/-10% value): {n_pairs:,}")
    print(f"Unique Late Orders Matched: {n_unique_late:,}")
    print(f"Average Late Delivery Review Score: {avg_late:.2f} stars")
    print(f"Average On-Time Delivery Review Score: {avg_ontime:.2f} stars")
    print(f"Controlled Score Penalty (Delta): {delta:.2f} stars (vs -1.72 unadjusted)")
    print("Conclusion: The -1.59 star penalty persists when controlling for distance,")
    print("            product category, and basket size. Randomization would still be")
    print("            required for causation; seller quality and seasonality remain confounds.")
    print("=" * 80)
else:
    print("[WAITING] data/processed/late_delivery_matched.csv not found.")


             CAUSAL INFERENCE: MATCHED-PAIR ANALYSIS (LATE VS ON-TIME)
Total Matched Pairs (same state, category/product, +/-10% value): 32,710
Unique Late Orders Matched: 3,309
Average Late Delivery Review Score: 2.61 stars
Average On-Time Delivery Review Score: 4.19 stars
Controlled Score Penalty (Delta): -1.59 stars (vs -1.72 unadjusted)
Conclusion: The -1.59 star penalty persists when controlling for distance,
            product category, and basket size. Randomization would still be
            required for causation; seller quality and seasonality remain confounds.
